# Week 2: Advanced Pandas Data Wrangling
## Notebook 13: Streaming Chunked ETL Pipelines & Memory-Bounded Processing

### 1. Conceptual Intuition & Theoretical Background
When ETL pipelines ingest multi-gigabyte files that exceed system memory, standard `pd.read_csv()` triggers fatal `MemoryError` crashes.

**Streaming Chunked ETL Architecture:**
- **Chunked Reader (`chunksize`):** Evaluates file streams as Python generators that yield fixed-size DataFrames (e.g. 10,000 rows at a time).
- **Memory Ceiling:** Memory consumption is bounded by the chunk size rather than total file size:
$$\text{Peak RAM} = O(\text{ChunkSize}) \ll O(\text{Total Rows})$$
- **Streaming Aggregations:** Computes rolling running statistics (running sums, frequencies, min/max) incrementally across chunks.

---

### 2. Scratch Python Implementation: Generator-Based Streaming ETL

In [ ]:
import pandas as pd
import numpy as np
import io

# Generate simulated 50,000-row CSV file in-memory
n_total_rows = 50000
csv_buffer = io.StringIO()
csv_buffer.write("transaction_id,merchant,amount,is_fraud\n")

for i in range(n_total_rows):
    m = "M_" + str(i % 5)
    amt = round(float(np.random.exponential(50.0)), 2)
    fraud = 1 if amt > 250.0 and np.random.rand() < 0.2 else 0
    csv_buffer.write(f"{i},{m},{amt},{fraud}\n")

csv_buffer.seek(0) # Reset buffer pointer

# Memory-Bounded Streaming Aggregator
chunk_size = 10000
running_counts = {}
running_total_amount = {}
total_fraud_count = 0
total_processed_rows = 0

# Stream processing loop using chunksize generator
for chunk in pd.read_csv(csv_buffer, chunksize=chunk_size):
    total_processed_rows += len(chunk)
    total_fraud_count += chunk['is_fraud'].sum()
    
    # Update running aggregates by merchant
    grouped = chunk.groupby('merchant')['amount'].agg(['count', 'sum'])
    for merchant, row in grouped.iterrows():
        running_counts[merchant] = running_counts.get(merchant, 0) + row['count']
        running_total_amount[merchant] = running_total_amount.get(merchant, 0.0) + row['sum']

# Finalize global metrics
summary_df = pd.DataFrame({
    'Transaction_Count': running_counts,
    'Total_Revenue': running_total_amount
})
summary_df['Average_Amount'] = (summary_df['Total_Revenue'] / summary_df['Transaction_Count']).round(2)

print("--- Streaming Chunked ETL Results ---")
print(f"Total Rows Streamed:     {total_processed_rows}")
print(f"Global Fraud Detections: {total_fraud_count}")
print("\nAggregated Summary by Merchant:")
print(summary_df)
print(f"\nProcessed in {n_total_rows // chunk_size} chunks with strictly bounded memory!")

### 3. Step-by-Step Code Walkthrough
1. **`chunksize` Parameter:** Converts `read_csv` into a generator producing discrete slices.
2. **Incremental Accumulation:** Running totals and counts are updated per chunk without storing intermediate rows.
3. **Bounded Memory:** Memory remains flat regardless of whether the file has 50,000 or 50,000,000 rows.

---

### 4. Industry Applications & Trade-offs
- **Cloud Ingestion Workers (AWS Lambda / Cloud Functions):** Processing massive files within strict memory limits (e.g. 512MB RAM).
- **Trade-off:** Global ranking operations (e.g., finding the true median) require two-pass algorithms or approximate sketches (T-Digest).

---

### 5. Top Interview Questions & Answers
**Q1: How do you compute the true median across a stream of data chunks without loading all rows into RAM?**
> **Answer:** You can use approximate quantile algorithms like the **T-Digest** or **Greenwald-Khanna** streaming sketch, or perform a two-pass binning histogram to pinpoint the median bucket.

**Q2: What is the purpose of passing `iterator=True` in `pd.read_csv`?**
> **Answer:** It returns a `TextFileReader` object allowing manual chunk retrieval via `reader.get_chunk(n)`, enabling dynamic variable-sized chunking logic.